# ✅ Solution 3: Trimmed Mean & Weighted Mean

**Approach**: Bandingkan robustness berbagai ukuran untuk data dengan bias, hitung IPK dengan weighted mean  
**Key Insight**: Trimmed mean 10% efektif menghilangkan review bias; weighted mean krusial jika bobot tidak seragam (SKS).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

np.random.seed(42)

# Data review restoran
skor_jujur = np.random.choice([3, 3, 3, 4, 4, 4, 4, 5, 5, 3], size=40)
skor_bias_rendah = np.array([1, 1, 1, 1, 1])
skor_bias_tinggi = np.array([5, 5, 5, 5, 5])
skor_semua = np.concatenate([skor_jujur, skor_bias_rendah, skor_bias_tinggi])
np.random.shuffle(skor_semua)

# Data mahasiswa
mahasiswa_A = pd.DataFrame({
    'mk': ['Data Mining', 'Statistika', 'Algoritma', 'Bahasa Inggris', 'Olahraga'],
    'sks': [4, 3, 4, 2, 1], 'nilai': [3.8, 3.5, 3.7, 4.0, 3.0]
})
mahasiswa_B = pd.DataFrame({
    'mk': ['Kalkulus', 'Fisika', 'Kimia', 'Pancasila', 'Olahraga'],
    'sks': [4, 4, 3, 2, 1], 'nilai': [2.5, 2.8, 3.0, 4.0, 4.0]
})
mahasiswa_C = pd.DataFrame({
    'mk': ['Machine Learning', 'Database', 'Web Dev', 'Etika', 'Seni'],
    'sks': [4, 3, 3, 2, 2], 'nilai': [3.9, 3.6, 3.5, 3.0, 3.0]
})

## 📖 Penjelasan Pendekatan

### Bagian A (Trimmed Mean):
- Review bias (hater & teman) ada di kedua ujung distribusi
- Trimmed mean membuang persentase tertentu dari ujung → mengeliminasi bias
- Kita bandingkan berbagai level trim untuk melihat mana yang paling "adil"

### Bagian B (Weighted Mean):
- IPK = Σ(SKS × Nilai) / Σ(SKS)
- MK dengan SKS tinggi harus berpengaruh lebih besar
- Mean biasa mengasumsikan semua MK sama pentingnya → SALAH

In [ ]:
# SOLUSI TODO 1: Berbagai ukuran central tendency
results_a = {
    'Mean biasa': np.mean(skor_semua),
    'Trimmed 5%': trim_mean(skor_semua, 0.05),
    'Trimmed 10%': trim_mean(skor_semua, 0.10),
    'Trimmed 20%': trim_mean(skor_semua, 0.20),
    'Median': np.median(skor_semua),
}

# Skor "jujur" sebagai benchmark
skor_jujur_mean = np.mean(skor_jujur)

print('=== Bagian A: Skor Review Restoran ===')
print(f'  Benchmark (mean skor jujur saja): {skor_jujur_mean:.3f}')
print()
print(f'  {"Ukuran":<15} | {"Nilai":<8} | {"Deviasi dari benchmark"}')
print(f'  {"-"*15} | {"-"*8} | {"-"*25}')
for name, val in results_a.items():
    dev = val - skor_jujur_mean
    print(f'  {name:<15} | {val:<8.3f} | {dev:+.3f}')
print()
print('💡 Trimmed mean 10% paling dekat dengan skor jujur!')
print('   Ia efektif menghilangkan 5 review hater + 5 review teman pemilik.')

In [ ]:
# SOLUSI TODO 2: IPK weighted mean
def hitung_ipk(df):
    mean_biasa = df['nilai'].mean()
    ipk = np.average(df['nilai'], weights=df['sks'])
    return mean_biasa, ipk

results_b = []
for nama, df in [('A', mahasiswa_A), ('B', mahasiswa_B), ('C', mahasiswa_C)]:
    mean_biasa, ipk = hitung_ipk(df)
    results_b.append({
        'mahasiswa': nama,
        'mean_biasa': round(mean_biasa, 4),
        'ipk_weighted': round(ipk, 4),
        'selisih': round(ipk - mean_biasa, 4),
    })

df_ipk = pd.DataFrame(results_b)
print('=== Bagian B: Perbandingan Mean Biasa vs IPK ===')
display(df_ipk)
print()
print('Interpretasi:')
print('  Mahasiswa A: IPK > Mean biasa → MK ber-SKS tinggi nilainya bagus')
print('  Mahasiswa B: IPK < Mean biasa → MK ber-SKS tinggi (Kalkulus 4SKS) nilainya RENDAH')
print('  Mahasiswa C: IPK > Mean biasa → MK ber-SKS tinggi (ML 4SKS) nilainya tinggi')
print()
print('💡 Mahasiswa B paling dirugikan oleh weighted mean karena')
print('   MK sulit (4 SKS) nilainya rendah, MK mudah (1-2 SKS) nilainya tinggi.')

In [ ]:
# SOLUSI TODO 3: Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Trimmed means restoran
names = list(results_a.keys())
values = list(results_a.values())
colors_a = ['#F44336', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']
axes[0].barh(names, values, color=colors_a, edgecolor='black', alpha=0.8)
axes[0].axvline(skor_jujur_mean, color='black', linestyle='--', lw=2, label=f'Benchmark = {skor_jujur_mean:.2f}')
axes[0].set_xlabel('Skor')
axes[0].set_title('Perbandingan Ukuran Central Tendency\n(Skor Review Restoran)', fontweight='bold')
axes[0].legend()
axes[0].set_xlim(2.5, 4.5)

# Subplot 2: Mean biasa vs IPK
x = range(3)
width = 0.35
axes[1].bar([i-width/2 for i in x], df_ipk['mean_biasa'], width, label='Mean Biasa', color='#FF5722', alpha=0.7, edgecolor='black')
axes[1].bar([i+width/2 for i in x], df_ipk['ipk_weighted'], width, label='IPK (Weighted)', color='#4CAF50', alpha=0.7, edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Mahasiswa A', 'Mahasiswa B', 'Mahasiswa C'])
axes[1].set_ylabel('Nilai')
axes[1].set_title('Mean Biasa vs IPK (Weighted Mean)', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(2.5, 4.2)

plt.tight_layout()
plt.show()

## 📌 Takeaways

- **Trimmed mean 10%** paling efektif untuk data dengan bias di kedua ujung (10% = 5 hater + 5 teman dari 50 total)
- **Weighted mean** wajib digunakan jika komponen punya bobot berbeda (IPK, indeks harga, survey)
- Mean biasa **over-simplifies** jika bobot tidak seragam
- Trimmed mean **lebih informatif** dari median karena masih menggunakan ~80% data
- Alternative: Winsorized mean (mengganti outlier dengan percentile, bukan membuang)